# Длинный эталон, MSE, бюджеты и независимые шумы

Исследуем DDIM, DPM-Solver++ 2M и Heun на google/ddpm-cifar10-32.
Как зависят качество и ошибка относительно длинной генерации от NFE и расписания?
Повторяется ли преимущество выбранных конфигураций на новых шумах?

Начинайте в новой сессии Colab. Восстановление данных доступно в CPU-сессии.
Для вычислений выберите
**Среда выполнения → Сменить среду выполнения → T4 GPU**.
При ограничении квоты GPU вычислительную часть можно выполнить после появления T4.

## Какие файлы загрузить

Три полных финальных ZIP из выполненного ноутбука 03. Подходят версии с визуальными
описаниями, названия которых заканчиваются на _reviewed.zip.

```text
ddim_uniform_t_nfe39_full_final_fid_000000-005000_8ce6f444.zip
dpmpp_2m_uniform_t_nfe39_full_final_fid_000000-005000_3376f091.zip
heun_uniform_t_nfe39_full_final_fid_000000-005000_a018e253.zip
```

Исходники и точный банк из 5000 шумов находятся внутри ZIP. После восстановления
устанавливаются зависимости. Для новых генераций загрузятся веса модели и данные FID.

## Фазы и ожидаемые результаты

| Фаза | Вычисления | Выход |
|---|---|---|
| convergence | DDIM-500/1000 и три метода при 39 NFE на 16 шумах | Проверка чувствительности к эталону |
| mse64 | Длинный эталон и 27 конфигураций на 64 шумах | MSE, ошибки отдельных изображений, графики |
| grid | 3 метода × 3 расписания × 3 бюджета, по 5000 изображений | FID, время, NFE, графики |
| mse1024 | Эталон и выбранные конфигурации на 1024 новых шумах | Подтверждение MSE |
| independent | Выбранные конфигурации на двух новых банках по 5000 шумов | Повторные FID и разброс |

Три готовых результата uniform_t/39 NFE используются при совпадении окружения.
При другом окружении создаётся новая сопоставимая группа. Длинный численный эталон
оценивается через различие DDIM-500 и DDIM-1000. FID и MSE интерпретируются отдельно.

## Как работать с блоками

Запускайте по одной ячейке. Один блок продвигает одну конфигурацию максимум на 16
батчей. Для длинного эталона блок ограничен одним батчем. После блока сохраняется
полный ZIP текущего запуска, скачивание выполняется следующей короткой ячейкой.
Дождитесь сохранения файла в загрузках браузера перед следующим блоком.

Сначала скачайте support ZIP с планом и всеми точными банками шумов. Обновите эту
копию после фиксации выбранных конфигураций. При потере сессии загрузите три исходных
ZIP, support ZIP и последний полный checkpoint каждого выполненного или начатого
запуска, который хотите восстановить. Завершённые задачи пропускаются.

План требует нескольких сессий T4. Ориентир полного расширения составляет 7–9 часов
вычислений, фактическое время уточняется по завершённым блокам и доступным ресурсам.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import json
import torch

PROJECT = Path('/content/diffusion-sampling')
SOURCE_BUNDLE = Path('/content/restored-source.zip')
print('CUDA доступна', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU', torch.cuda.get_device_name(0))
print('Восстановление файлов доступно в текущей сессии')

## 1. Проверить и восстановить три архива

Сначала определяются функции восстановления. Затем выберите три полных финальных
ZIP методов. Контрольные суммы проверяются перед восстановлением. Конфликтующие
существующие файлы вызывают ошибку с объяснением.

In [ ]:
"""Restore a full experiment export using only Python's standard library."""

from io import BytesIO
import hashlib
import json
from pathlib import Path, PurePosixPath
import shutil
import tempfile
from zipfile import ZipFile


def _relative_path(name):
    path = PurePosixPath(name)
    if not name or path.is_absolute() or ".." in path.parts or "\\" in name:
        raise ValueError(f"Unsafe archive path: {name}")
    return path


def _destination(root, name):
    path = root.joinpath(*_relative_path(name).parts)
    if not path.resolve().is_relative_to(root.resolve()):
        raise ValueError(f"Path leaves the project: {name}")
    return path


def _source_fingerprint(source):
    prefix = "diffusion-sampling/src/diffusion_sampling/"
    records = []
    for name in sorted(source.namelist()):
        relative = name.removeprefix(prefix)
        if name.startswith(prefix) and relative.endswith(".py") and "/" not in relative:
            records.append((relative, hashlib.sha256(source.read(name)).hexdigest()))
    if not records:
        raise ValueError("Source ZIP has no experiment package")
    return hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()


def restore_full_archive(archive_path, project, restore_source=False, noise_equal=None):
    """Verify a full ZIP and restore its run, exact noise and optional source.

    Existing runs must be identical to the uploaded snapshot. Conflicting files
    cause an error before restoration. Run directories are published atomically.
    """
    project = Path(project).resolve()
    with ZipFile(archive_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("ZIP contains duplicate entry names")
        for name in names:
            _relative_path(name)
        metadata = json.loads(archive.read("export_manifest.json"))
        hashes = metadata["files_sha256"]
        if set(names) != set(hashes) | {"export_manifest.json"}:
            raise ValueError("ZIP entries differ from the export manifest")
        for name, expected in hashes.items():
            if hashlib.sha256(archive.read(name)).hexdigest() != expected:
                raise ValueError(f"Checksum mismatch: {name}")
        if metadata["kind"] != "full" or metadata["start_image"] != 0:
            raise ValueError("Upload a full archive starting at image zero")
        manifest = json.loads(archive.read("run/manifest.json"))
        summary = json.loads(archive.read("run/summary.json"))
        experiment_id = manifest["experiment_id"]
        count = summary["generated_images"]
        if metadata["experiment_id"] != experiment_id or summary["experiment_id"] != experiment_id:
            raise ValueError("Experiment identifiers differ")
        if metadata["end_image_exclusive"] != count or metadata["images_in_archive"] != count:
            raise ValueError("Snapshot image counts differ")
        expected_images = {f"run/images/{index:08d}.png" for index in range(count)}
        if {name for name in names if name.startswith("run/images/")} != expected_images:
            raise ValueError("Snapshot PNG sequence has gaps or extra entries")
        config = manifest["config"]
        expected_batches = {
            f"run/batches/{index:08d}.pt"
            for index in range(0, count, config["batch_size"])
        }
        if {name for name in names if name.startswith("run/batches/")} != expected_batches:
            raise ValueError("Snapshot batch sequence has gaps or extra entries")
        run_relative = _relative_path(config["output_dir"])
        noise_relative = _relative_path(config["noise_path"])
        if run_relative.parts[0] != "results" or noise_relative.parts[0] != "data":
            raise ValueError("Expected results/ and data/ paths in the saved config")
        run_dir = _destination(project, run_relative.as_posix())
        noise_path = _destination(project, noise_relative.as_posix())
        noise_bytes = archive.read("initial_noise.pt")
        if noise_path.exists() and noise_path.read_bytes() != noise_bytes:
            if noise_equal is None or not noise_equal(noise_path, BytesIO(noise_bytes)):
                raise ValueError("An existing noise bank differs from the uploaded bank")
        source_bytes = archive.read("source.zip")
        source_files = {}
        with ZipFile(BytesIO(source_bytes)) as source:
            if len(source.namelist()) != len(set(source.namelist())):
                raise ValueError("Source ZIP contains duplicate entry names")
            if _source_fingerprint(source) != manifest["environment"]["source_sha256"]:
                raise ValueError("Source ZIP differs from the generation source hash")
            for entry in source.infolist():
                relative = _relative_path(entry.filename)
                if relative.parts[0] != "diffusion-sampling" or len(relative.parts) < 2:
                    raise ValueError("Unexpected source ZIP root")
                if not entry.is_dir():
                    source_files[PurePosixPath(*relative.parts[1:]).as_posix()] = source.read(entry)
        if restore_source:
            for name, data in source_files.items():
                path = _destination(project, name)
                if path.exists() and path.read_bytes() != data:
                    raise ValueError(f"Existing source differs: {name}")
        else:
            package = project / "src/diffusion_sampling"
            records = [(path.name, hashlib.sha256(path.read_bytes()).hexdigest())
                       for path in sorted(package.glob("*.py"))]
            fingerprint = hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()
            if fingerprint != manifest["environment"]["source_sha256"]:
                raise ValueError("Restore the matching project source before this run")
        run_files = {name.removeprefix("run/"): name for name in names if name.startswith("run/")}
        if run_dir.exists():
            existing = {path.relative_to(run_dir).as_posix()
                        for path in run_dir.rglob("*") if path.is_file()}
            if existing != set(run_files):
                raise ValueError("Existing run differs from the uploaded snapshot")
            for relative, name in run_files.items():
                if _destination(run_dir, relative).read_bytes() != archive.read(name):
                    raise ValueError(f"Existing run differs: {relative}")
        project.mkdir(parents=True, exist_ok=True)
        if restore_source:
            for name, data in source_files.items():
                path = _destination(project, name)
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(data)
        noise_path.parent.mkdir(parents=True, exist_ok=True)
        if not noise_path.exists():
            with tempfile.NamedTemporaryFile(dir=noise_path.parent, delete=False) as stream:
                temporary_noise = Path(stream.name)
                stream.write(noise_bytes)
            temporary_noise.replace(noise_path)
        if not run_dir.exists():
            run_dir.parent.mkdir(parents=True, exist_ok=True)
            staging = Path(tempfile.mkdtemp(prefix=".restore-", dir=run_dir.parent))
            try:
                for relative, name in run_files.items():
                    path = _destination(staging, relative)
                    path.parent.mkdir(parents=True, exist_ok=True)
                    path.write_bytes(archive.read(name))
                staging.replace(run_dir)
            finally:
                if staging.exists():
                    shutil.rmtree(staging)
    return {"run_dir": run_dir, "noise_path": noise_path, "manifest": manifest,
            "summary": summary, "source_bytes": source_bytes}

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 3, 'Выберите три полных ZIP методов из ноутбука 03'
archived_runs = []
seen_methods = set()
for index, (name, data) in enumerate(uploaded.items()):
    path = Path('/content') / f'input_method_{index}.zip'
    path.write_bytes(data)
    restored = restore_full_archive(
        path, PROJECT, restore_source=(index == 0),
        noise_equal=lambda left, right: torch.equal(
            torch.load(left, map_location='cpu', weights_only=True)['noise'],
            torch.load(right, map_location='cpu', weights_only=True)['noise'],
        ),
    )
    config = restored['manifest']['config']
    assert config['sampler'] in {'ddim', 'dpmpp_2m', 'heun'}
    assert config['schedule'] == 'uniform_t' and not config['clip_sample']
    assert restored['summary']['nfe_per_image'] == 39
    assert restored['summary']['generated_images'] == 5000
    assert restored['summary']['baseline_metrics_complete']
    seen_methods.add(config['sampler'])
    archived_runs.append(restored['run_dir'])
    if index == 0:
        SOURCE_BUNDLE.write_bytes(restored['source_bytes'])
    print(config['sampler'], 'FID', restored['summary']['fid'])
assert seen_methods == {'ddim', 'dpmpp_2m', 'heun'}
del uploaded, data

## 2. Установить зависимости и определить функции исследования

Устанавливаем сохранённый пакет эксперимента и matplotlib для научных графиков.
Функции следующей ячейки встроены в ноутбук. Все формулы сэмплеров, проверки
запусков, FID и экспорт используются из восстановленной версии пакета.

In [ ]:
%pip install -q -e /content/diffusion-sampling matplotlib
%cd /content/diffusion-sampling
import sys
sys.path.insert(0, str(PROJECT / 'src'))
os.environ.setdefault('TORCH_HOME', str(PROJECT / '.cache/torch'))
os.environ.setdefault('HF_HOME', str(PROJECT / '.cache/huggingface-home'))
from IPython.display import Image, display

In [ ]:
"""Colab orchestration for references, matched budgets and independent noise banks.

This script is embedded in notebook 04. The archived experiment package supplies
the numerical methods, generation, FID, comparison and portable run exports.
"""

import csv
import gc
import hashlib
from io import BytesIO
import json
from pathlib import Path
import re
from uuid import uuid4
from zipfile import ZIP_DEFLATED, ZipFile

import torch

from diffusion_sampling.config import RunConfig
from diffusion_sampling.evaluation import evaluate_fid
from diffusion_sampling.experiment import run_experiment
from diffusion_sampling.exports import CheckpointExporter
from diffusion_sampling.noise import ensure_noise, load_noise
from diffusion_sampling.runtime import configure_runtime, environment
from diffusion_sampling.schedules import points_for_budget
from diffusion_sampling.storage import file_sha256, read_json, write_json
from diffusion_sampling.study import _load_run, compare_runs


METHODS = ("ddim", "dpmpp_2m", "heun")
SCHEDULES = ("uniform_t", "log_sigma", "karras")


def task_label(method, schedule, budget):
    return f"{method}_{schedule}_nfe{budget}"


def _task(plan, phase, label, method, schedule, budget, bank_name, count, fid=False):
    bank = plan["banks"][bank_name]
    config = RunConfig(
        sampler=method, schedule=schedule, num_steps=points_for_budget(method, budget),
        num_samples=count, batch_size=plan["batch_size"], device=plan["device"],
        clip_sample=False, noise_path=bank["path"], noise_seed=bank["seed"],
        purpose="baseline" if fid else "pilot",
        output_dir=f"results/{plan['name']}/{phase}/{label}",
    )
    return {"label": label, "config": config.to_dict(), "fid_required": fid}


def prepare_plan(project, name, archived_runs=(), device="cuda", budgets=(19, 39, 79),
                 selection_budget=39, fid_count=5000, mse_count=64, confirm_count=1024,
                 debug_count=16, reference_budgets=(500, 1000), batch_size=64):
    """Create a fixed design, or load the same design after session recovery."""
    if not re.fullmatch(r"[A-Za-z0-9_-]+", name):
        raise ValueError("Use letters, numbers, underscores or hyphens for the study name")
    if selection_budget not in budgets:
        raise ValueError("The selection budget must be included in the grid")
    for budget in budgets:
        points_for_budget("heun", budget)
    project = Path(project)
    configure_runtime(4)
    current = environment(torch.device(device))
    design = {"budgets": list(budgets), "selection_budget": selection_budget,
              "fid_count": fid_count, "mse_count": mse_count, "confirm_count": confirm_count,
              "debug_count": debug_count, "reference_budgets": list(reference_budgets),
              "batch_size": batch_size, "device": device}
    path = project / f"results/{name}/extended_plan.json"
    if path.exists():
        plan = read_json(path)
        if plan["design"] != design or plan["environment"] != current:
            raise ValueError("Design or environment changed. Choose a new STUDY_NAME.")
        return plan
    prefix = f"data/noise/{name}"
    plan = {"schema_version": 1, "name": name, "design": design, "environment": current,
            "device": device, "batch_size": batch_size, "selection": [],
            "banks": {
                "debug": {"path": f"{prefix}/debug.pt", "seed": 10000, "count": debug_count},
                "mse64": {"path": f"{prefix}/mse64.pt", "seed": 30000, "count": mse_count},
                "mse1024": {"path": f"{prefix}/mse1024.pt", "seed": 40000, "count": confirm_count},
                "fid": {"path": "data/noise/fid.pt", "seed": 50000, "count": fid_count},
                "fid60000": {"path": f"{prefix}/fid60000.pt", "seed": 60000, "count": fid_count},
                "fid70000": {"path": f"{prefix}/fid70000.pt", "seed": 70000, "count": fid_count},
            }, "phases": {"convergence": [], "mse64": [], "grid": [],
                          "mse1024": [], "independent": []}}
    intervals = sorted((bank["seed"], bank["seed"] + bank["count"]) for bank in plan["banks"].values())
    if any(left[1] > right[0] for left, right in zip(intervals, intervals[1:])):
        raise ValueError("Noise-bank seed ranges overlap")
    for bank in plan["banks"].values():
        payload = ensure_noise(project / bank["path"], bank["count"], bank["seed"])
        if payload["count"] != bank["count"]:
            raise ValueError("This study requires the exact saved bank size")
        bank["sha256"] = payload["sha256"]
    for budget in reference_budgets:
        plan["phases"]["convergence"].append(_task(
            plan, "convergence", f"reference_{budget}", "ddim", "uniform_t",
            budget, "debug", debug_count))
    for method in METHODS:
        plan["phases"]["convergence"].append(_task(
            plan, "convergence", task_label(method, "uniform_t", selection_budget),
            method, "uniform_t", selection_budget, "debug", debug_count))
    plan["phases"]["mse64"].append(_task(
        plan, "mse64", "reference", "ddim", "uniform_t", reference_budgets[-1], "mse64", mse_count))
    old_runs = {}
    for directory in archived_runs:
        manifest, summary, _ = _load_run(directory)
        config = RunConfig(**manifest["config"])
        if (manifest["environment"] == current and not config.clip_sample
                and config.schedule == "uniform_t" and config.num_samples == fid_count
                and config.batch_size == batch_size and config.noise_seed == 50000
                and manifest["noise_sha256"] == plan["banks"]["fid"]["sha256"]
                and summary["baseline_metrics_complete"] and config.expected_nfe in budgets):
            old_runs[task_label(config.sampler, config.schedule, config.expected_nfe)] = config.to_dict()
    for method in METHODS:
        for schedule in SCHEDULES:
            for budget in budgets:
                label = task_label(method, schedule, budget)
                plan["phases"]["mse64"].append(_task(
                    plan, "mse64", label, method, schedule, budget, "mse64", mse_count))
                task = _task(plan, "grid", label, method, schedule, budget, "fid", fid_count, fid=True)
                if label in old_runs:
                    task["config"] = old_runs[label]
                    task["reused"] = True
                plan["phases"]["grid"].append(task)
    write_json(path, plan)
    return plan


def _summary(project, plan, task):
    directory = Path(project) / task["config"]["output_dir"]
    if not (directory / "manifest.json").exists():
        return None
    manifest = read_json(directory / "manifest.json")
    if manifest["config"] != task["config"] or manifest["environment"] != plan["environment"]:
        raise ValueError("Saved config or runtime differs. Restore its original environment or use a new study name.")
    return read_json(directory / "summary.json")


def phase_progress(project, plan, phase):
    rows = []
    for task in plan["phases"][phase]:
        summary = _summary(project, plan, task)
        done = bool(summary and summary["generation_complete"]
                    and (not task["fid_required"] or summary["baseline_metrics_complete"]))
        rows.append({"label": task["label"], "done": done,
                     "images": summary["generated_images"] if summary else 0,
                     "requested": task["config"]["num_samples"]})
    return rows


def run_next_block(project, plan, phase, source_bundle, max_batches=16,
                   runner=run_experiment, evaluator=evaluate_fid):
    """Advance one task by bounded batches and publish a full portable snapshot."""
    project = Path(project)
    configure_runtime(4)
    if environment(torch.device(plan["device"])) != plan["environment"]:
        raise ValueError("Runtime changed. Choose a new study name for new measurements.")
    tasks = plan["phases"][phase]
    if not tasks:
        raise ValueError("Run the grid and lock selection before this phase")
    for task, progress in zip(tasks, phase_progress(project, plan, phase)):
        if progress["done"]:
            continue
        config = RunConfig(**task["config"])
        bank = next(bank for bank in plan["banks"].values() if bank["path"] == config.noise_path)
        if load_noise(project / config.noise_path)["sha256"] != bank["sha256"]:
            raise ValueError("Noise bank changed. Restore the exact bank from the support ZIP.")
        print("Task", phase, task["label"], flush=True)
        block_batches = min(max_batches, 1) if config.expected_nfe >= 500 else max_batches
        result = runner(config, root=project, max_batches=block_batches)
        directory = project / config.output_dir
        if result["generation_complete"] and task["fid_required"] and not result["baseline_metrics_complete"]:
            gc.collect()
            if plan["device"] == "cuda":
                torch.cuda.empty_cache()
            evaluator(directory, project / "data/cifar10", project / ".cache/fidelity",
                      device=plan["device"], download=True)
        logs = directory / "logs"
        logs.mkdir(exist_ok=True)
        write_json(logs / "extended_plan.log", plan)
        exporter = CheckpointExporter(project, project / f"results/exports/{plan['name']}",
                                      source_bundle=source_bundle)
        result = read_json(directory / "summary.json")
        stage = "final_fid" if result["baseline_metrics_complete"] else "progress"
        output = exporter.full(directory, stage=f"{phase}_{stage}")
        print("Images", result["generated_images"], "/", config.num_samples, flush=True)
        print("Full checkpoint", output, flush=True)
        return output
    print("Phase complete", phase)
    return None


def _write_csv(path, rows):
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def _mse_details(project, tasks, reference, output):
    _, _, reference_samples = _load_run(reference)
    rows = []
    for task in tasks:
        directory = Path(project) / task["config"]["output_dir"]
        _, _, samples = _load_run(directory)
        errors = (samples - reference_samples).square().flatten(1).mean(1).double()
        rows.extend({"label": task["label"], "image_index": index, "mse": float(error)}
                    for index, error in enumerate(errors))
    _write_csv(output, rows)


def _plots(report, output):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    rows = report["rows"]
    has_fid = all(row["fid"] is not None for row in rows)
    metric = "fid" if has_fid else "mse_to_reference"
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for method in METHODS:
        for schedule in SCHEDULES:
            selected = sorted((row for row in rows if row["sampler"] == method
                               and row["schedule"] == schedule), key=lambda row: row["nfe"])
            if not selected:
                continue
            label = f"{method} / {schedule}"
            axes[0].plot([row["nfe"] for row in selected], [row[metric] for row in selected],
                         marker="o", label=label)
            axes[1].plot([row["seconds_per_image"] for row in selected],
                         [row[metric] for row in selected], marker="o", label=label)
    axes[0].set_xlabel("NFE per image")
    axes[1].set_xlabel("Denoising seconds per image")
    for axis in axes:
        axis.set_ylabel("FID-5k / real-50k" if has_fid else "MSE to DDIM reference")
        axis.grid(alpha=0.25)
    axes[1].legend(fontsize=7, loc="best")
    fig.tight_layout()
    fig.savefig(output, dpi=180)
    plt.close(fig)


def write_phase_report(project, plan, phase):
    project = Path(project)
    progress = phase_progress(project, plan, phase)
    if not progress or not all(row["done"] for row in progress):
        raise ValueError("Finish all tasks in this phase before generating its report")
    directory = project / f"results/{plan['name']}/reports"
    directory.mkdir(parents=True, exist_ok=True)
    tasks = plan["phases"][phase]
    runs = [project / task["config"]["output_dir"] for task in tasks]
    if phase == "convergence":
        low, high = runs[:2]
        compare_runs([low], directory / "reference_convergence.json", reference=high)
        report = compare_runs(runs[2:], directory / "convergence.json", reference=high)
        compare_runs(runs[2:], directory / "convergence_to_500.json", reference=low)
        _mse_details(project, tasks[2:], high, directory / "convergence_per_image.csv")
    elif phase in ("mse64", "mse1024"):
        report = compare_runs(runs[1:], directory / f"{phase}.json", reference=runs[0])
        _mse_details(project, tasks[1:], runs[0], directory / f"{phase}_per_image.csv")
        _plots(report, directory / f"{phase}_curves.png")
    elif phase == "grid":
        report = compare_runs(runs, directory / "grid.json")
        _plots(report, directory / "grid_curves.png")
    else:
        combined = []
        for seed in (60000, 70000):
            selected = [project / task["config"]["output_dir"] for task in tasks
                        if task["config"]["noise_seed"] == seed]
            result = compare_runs(selected, directory / f"independent_{seed}.json")
            combined.extend({**row, "noise_seed": seed} for row in result["rows"])
        aggregate = []
        for method, schedule, budget in sorted({(r["sampler"], r["schedule"], r["nfe"]) for r in combined}):
            values = [r["fid"] for r in combined if (r["sampler"], r["schedule"], r["nfe"]) == (method, schedule, budget)]
            aggregate.append({"sampler": method, "schedule": schedule, "nfe": budget,
                              "banks": len(values), "mean_fid": sum(values) / len(values),
                              "min_fid": min(values), "max_fid": max(values)})
        report = {"rows": combined, "aggregate": aggregate, "selection": plan["selection"]}
        write_json(directory / "independent.json", report)
        _write_csv(directory / "independent.csv", aggregate)
    return report


def lock_selection(project, plan):
    """Choose up to six fixed configs on the primary bank before confirmation."""
    if plan["selection"]:
        return plan
    report = write_phase_report(project, plan, "grid")
    budget = plan["design"]["selection_budget"]
    selected = set()
    for method in METHODS:
        rows = [row for row in report["rows"] if row["sampler"] == method and row["nfe"] == budget]
        best = min(rows, key=lambda row: (row["fid"], row["schedule"]))
        selected.add((method, "uniform_t", budget))
        selected.add((method, best["schedule"], budget))
    plan["selection"] = [task_label(*values) for values in sorted(selected)]
    plan["selection_source_sha256"] = file_sha256(Path(project) / f"results/{plan['name']}/reports/grid.json")
    count = plan["design"]["confirm_count"]
    high = plan["design"]["reference_budgets"][-1]
    plan["phases"]["mse1024"] = [_task(plan, "mse1024", "reference", "ddim", "uniform_t", high, "mse1024", count)]
    for method, schedule, budget in sorted(selected):
        label = task_label(method, schedule, budget)
        plan["phases"]["mse1024"].append(_task(plan, "mse1024", label,
                                              method, schedule, budget, "mse1024", count))
        for seed in (60000, 70000):
            plan["phases"]["independent"].append(_task(
                plan, "independent", f"{label}_seed{seed}", method, schedule,
                budget, f"fid{seed}", plan["design"]["fid_count"], fid=True))
    write_json(Path(project) / f"results/{plan['name']}/extended_plan.json", plan)
    return plan


def support_backup(project, plan, source_bundle):
    """Save the fixed design and all exact noise banks for another Colab session."""
    project = Path(project)
    directory = project / f"results/exports/{plan['name']}"
    directory.mkdir(parents=True, exist_ok=True)
    output = directory / f"{plan['name']}_support_{uuid4().hex[:8]}.zip"
    hashes = {}
    with ZipFile(output, "w", compression=ZIP_DEFLATED) as archive:
        data = (json.dumps(plan, ensure_ascii=False, indent=2) + "\n").encode()
        archive.writestr("extended_plan.json", data)
        hashes["extended_plan.json"] = hashlib.sha256(data).hexdigest()
        for bank in plan["banks"].values():
            payload = load_noise(project / bank["path"])
            if payload["sha256"] != bank["sha256"]:
                raise ValueError("Noise bank changed")
            data = (project / bank["path"]).read_bytes()
            archive.writestr(bank["path"], data)
            hashes[bank["path"]] = hashlib.sha256(data).hexdigest()
        data = Path(source_bundle).read_bytes()
        archive.writestr("source.zip", data)
        hashes["source.zip"] = hashlib.sha256(data).hexdigest()
        archive.writestr("support_manifest.json", json.dumps({"files_sha256": hashes}))
    return output


def restore_support(path, project, current_environment):
    project = Path(project)
    with ZipFile(path) as archive:
        metadata = json.loads(archive.read("support_manifest.json"))
        hashes = metadata["files_sha256"]
        if len(archive.namelist()) != len(set(archive.namelist())) or set(archive.namelist()) != set(hashes) | {"support_manifest.json"}:
            raise ValueError("Support ZIP entries differ from its manifest")
        for name, expected in hashes.items():
            if hashlib.sha256(archive.read(name)).hexdigest() != expected:
                raise ValueError(f"Support checksum mismatch: {name}")
        plan = json.loads(archive.read("extended_plan.json"))
        if plan["environment"] != current_environment:
            raise ValueError("Support environment differs. Create a new study name for this runtime.")
        if not re.fullmatch(r"[A-Za-z0-9_-]+", plan["name"]):
            raise ValueError("Unsafe study name")
        bank_paths = {bank["path"] for bank in plan["banks"].values()}
        for tasks in plan["phases"].values():
            for task in tasks:
                config = RunConfig(**task["config"])
                output = project / config.output_dir
                if (not output.resolve().is_relative_to(project.resolve())
                        or not config.output_dir.startswith("results/")
                        or config.noise_path not in bank_paths):
                    raise ValueError("Unsafe task path in support plan")
        for bank in plan["banks"].values():
            destination = project / bank["path"]
            if not destination.resolve().is_relative_to(project.resolve()) or not bank["path"].startswith("data/"):
                raise ValueError("Unsafe noise bank path")
            data = archive.read(bank["path"])
            incoming = load_noise(BytesIO(data))
            if incoming["sha256"] != bank["sha256"] or incoming["count"] != bank["count"] or incoming["seed"] != bank["seed"]:
                raise ValueError("Saved noise differs from the support plan")
            if destination.exists() and destination.read_bytes() != data:
                existing = load_noise(destination)
                if existing["sha256"] != bank["sha256"]:
                    raise ValueError("An existing noise bank differs")
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(data)
        write_json(project / f"results/{plan['name']}/extended_plan.json", plan)
    return plan


def report_backup(project, plan, source_bundle):
    project = Path(project)
    directory = project / f"results/exports/{plan['name']}"
    directory.mkdir(parents=True, exist_ok=True)
    output = directory / f"{plan['name']}_reports_{uuid4().hex[:8]}.zip"
    with ZipFile(output, "w", compression=ZIP_DEFLATED) as archive:
        state = project / f"results/{plan['name']}/extended_plan.json"
        archive.write(state, "extended_plan.json")
        reports = project / f"results/{plan['name']}/reports"
        for path in sorted(reports.glob('*')):
            if path.is_file():
                archive.write(path, Path("reports") / path.name)
        archive.write(source_bundle, "source.zip")
    return output

## 3. Параметры и восстановление прогресса

Для первоначального запуска оставьте RESTORE_PROGRESS=False. Для продолжения
задайте True. STUDY_NAME должен соответствовать сохранённому support ZIP.
При смене окружения используйте новое имя и создайте новую группу измерений.

Следующая ячейка требует T4. Все задачи имеют float32, clipping=False, batch size 64,
одинаковые крайние уровни шума и один сохранённый банк внутри каждого сравнения.

In [ ]:
STUDY_NAME = 'extended_study_v1'
RESTORE_PROGRESS = False
DOWNLOAD_RESULTS = True
MAX_NEW_BATCHES = 16

assert torch.cuda.is_available(), 'Для вычислительных фаз подключите GPU T4'
assert 'T4' in torch.cuda.get_device_name(0), 'Выберите T4 для этой группы измерений'
configure_runtime(4)
current_environment = environment(torch.device('cuda'))

def same_saved_noise(left, right):
    existing, incoming = load_noise(left), load_noise(right)
    count = incoming['count']
    return existing['count'] >= count and torch.equal(existing['noise'][:count], incoming['noise'])

if RESTORE_PROGRESS:
    progress_files = files.upload()
    support_files = []
    run_files = []
    for name, data in progress_files.items():
        path = Path('/content') / Path(name).name
        path.write_bytes(data)
        with ZipFile(path) as archive:
            if 'support_manifest.json' in archive.namelist():
                support_files.append(path)
            else:
                run_files.append(path)
    assert len(support_files) == 1, 'Выберите один последний support ZIP и checkpoint ZIP запусков'
    saved_plan = restore_support(support_files[0], PROJECT, current_environment)
    assert saved_plan['name'] == STUDY_NAME
    allowed_configs = [task['config'] for tasks in saved_plan['phases'].values() for task in tasks]
    for path in run_files:
        with ZipFile(path) as archive:
            saved_manifest = json.loads(archive.read('run/manifest.json'))
        assert saved_manifest['config'] in allowed_configs, 'ZIP относится к другой группе'
        assert saved_manifest['environment'] == current_environment
        restored = restore_full_archive(path, PROJECT, noise_equal=same_saved_noise)
        if restored['summary']['generation_complete']:
            _load_run(restored['run_dir'])
        print('Восстановлено', restored['run_dir'], restored['summary']['generated_images'])
    del progress_files

plan = prepare_plan(PROJECT, STUDY_NAME, archived_runs=archived_runs)
for phase in plan['phases']:
    rows = phase_progress(PROJECT, plan, phase)
    print(phase, 'готово', sum(row['done'] for row in rows), '/', len(rows))
support_zip = support_backup(PROJECT, plan, SOURCE_BUNDLE)
print('Support ZIP', support_zip)

In [ ]:
def download_one(path):
    if path is None:
        return
    print('Архив готов', path, f'{path.stat().st_size / 2**20:.1f} MiB')
    if DOWNLOAD_RESULTS:
        files.download(str(path))
    print('Дождитесь завершения скачивания в браузере')

download_one(support_zip)

## 4. Тесты и численная сверка реализаций

Ожидается успешное завершение тестов. Затем выполняется короткая численная сверка
с Diffusers на настоящей модели в текущем окружении T4. Сравниваются промежуточные
состояния и фактический NFE. Веса загрузятся при первом обращении к модели.

In [ ]:
import subprocess
tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                       cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(tests.stdout)
tests.check_returncode()
test_log = PROJECT / f'results/{STUDY_NAME}/reports/tests.log'
test_log.parent.mkdir(parents=True, exist_ok=True)
test_log.write_text(tests.stdout)
verification = subprocess.run(
    [sys.executable, 'scripts/verify_samplers.py', '--device', 'cuda',
     '--output', f'results/{STUDY_NAME}/reports/verification.json'],
    cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(verification.stdout)
(test_log.parent / 'verification.log').write_text(verification.stdout)
verification.check_returncode()
assert json.loads((test_log.parent / 'verification.json').read_text())['passed']

## 5. Выполнение выбранной фазы

Сначала PHASE='convergence'. Затем последовательно mse64 и grid.
После grid сформируйте отчёт и зафиксируйте выбор в разделе 7.
После фиксации доступны mse1024 и independent.

Для каждой фазы повторяйте две ячейки блока и скачивания, пока все строки
не получат done=True. Строки прогресса показывают число изображений каждой задачи.
Следующий блок автоматически выбирает первую незавершённую задачу.

Для Heun бюджеты 19/39/79 соответствуют 10/20/40 узлам. Для DDIM и DPM-Solver++
число узлов равно NFE. При завершении генерации FID-задачи автоматически вычисляется
FID по протоколу generated-5k / CIFAR-10 train-50k. Время FID учитывается отдельно.

In [ ]:
PHASE = 'convergence'  # Затем mse64, grid, mse1024, independent.
for row in phase_progress(PROJECT, plan, PHASE):
    print(row)

In [ ]:
latest_checkpoint = run_next_block(
    PROJECT, plan, PHASE, SOURCE_BUNDLE, max_batches=MAX_NEW_BATCHES,
)
for row in phase_progress(PROJECT, plan, PHASE):
    print(row)

In [ ]:
download_one(latest_checkpoint)

## 6. Отчёт завершённой фазы

После done=True у всех задач выполните следующую ячейку. Она проверяет условия
сравнения, сохраняет JSON/CSV/сетки и графики. Для MSE сохраняются ошибки отдельных
изображений. Для convergence выводится также MSE DDIM-500 к DDIM-1000.

Посмотрите величину ошибки между эталонами и порядок коротких методов при обоих
эталонах. Заметная чувствительность к эталону должна быть отражена в выводах.

In [ ]:
phase_report = write_phase_report(PROJECT, plan, PHASE)
print(json.dumps(phase_report, ensure_ascii=False, indent=2))
reports_dir = PROJECT / f'results/{STUDY_NAME}/reports'
curve = reports_dir / f'{PHASE}_curves.png'
if curve.exists():
    display(Image(filename=str(curve)))
if PHASE == 'convergence':
    print('Ошибка между эталонами')
    print((reports_dir / 'reference_convergence.json').read_text())
reports_zip = report_backup(PROJECT, plan, SOURCE_BUNDLE)

In [ ]:
download_one(reports_zip)

## 7. Зафиксировать конфигурации перед независимой проверкой

Выполняйте после полного grid. Для каждого метода выбирается лучшее расписание
при 39 NFE по основному банку 50000–54999. Добавляется uniform_t каждого метода.
Совпадения объединяются. Получается от трёх до шести фиксированных конфигураций.

После фиксации выполняются mse1024 на шумах 40000–41023 и independent на двух
банках 60000–64999 и 70000–74999. Повторно скачайте support ZIP, чтобы сохранить
выбранные конфигурации и точные банки перед подтверждающими экспериментами.

In [ ]:
plan = lock_selection(PROJECT, plan)
print('Зафиксированы конфигурации', plan['selection'])
support_zip = support_backup(PROJECT, plan, SOURCE_BUNDLE)
for phase in ['mse1024', 'independent']:
    print(phase, 'задач', len(plan['phases'][phase]))

In [ ]:
download_one(support_zip)

Вернитесь в раздел 5 и выполните сначала mse1024, затем independent.
После каждой завершённой фазы скачайте отчёт из раздела 6.

## Какие результаты сохранить

1. Последний support ZIP после фиксации конфигураций.
2. Последний полный checkpoint каждого запуска для восстановления данных.
3. Последний reports ZIP с JSON, CSV, сетками и графиками.
4. Выполненный ноутбук через **Файл → Скачать → Скачать IPYNB**.

В independent.csv записаны средний FID и минимальное/максимальное значения по двум
новым банкам. Каждый подтверждающий прогон сохранён отдельно. Основной банк служит
для выбора конфигураций. Устойчивость оценивается по новым банкам.

Визуальные наблюдения и проверенные результаты uniform_t/39 NFE доступны в репозитории
в docs/study_uniform39_results.md и docs/visual_reviews. Подбор собственных опорных
точек расписания требует следующей реализации после сравнения стандартных сеток.